In [70]:
import mysql.connector
import numpy as np
import pandas as pd
import astropy.io.fits as fits
from matplotlib import pyplot as plt
import tiledb
import os

HEIGHT = 256
WIDTH = 384

In [71]:
conn = mysql.connector.connect(
    user='root',
    password='',
    unix_socket='/tmp/mysql_database_dev.sock',
    database='heat_db'
)
if conn.is_connected():
    print("Connected to MySQL database")
else:
    print("Failed to connect to MySQL database")

Connected to MySQL database


In [72]:
filenames = []
cur = conn.cursor(dictionary=True)
cur.execute("SELECT DISTINCT img_file FROM pix01")
for fetched_line in cur.fetchall():
    filenames.append(fetched_line['img_file'])
print(len(filenames))

9516


In [73]:
def fetch_img_file(filename):
    DN = np.zeros((HEIGHT, WIDTH))
    Modified_DN = np.zeros((HEIGHT, WIDTH))
    Mask = np.zeros((HEIGHT, WIDTH))
    for i in range(96):
        pix_num = f'pix{i + 1:02d}'
        query__for_fetching_data = f"SELECT * FROM {pix_num} WHERE img_file = %s"
        cur.execute(query__for_fetching_data, (filename,))
        for fetched_line in cur.fetchall():
            x = fetched_line['x']
            y = fetched_line['y']
            pixel = fetched_line['pixel']
            pixel_modified = fetched_line['pixel_modified']
            mask = fetched_line['mask']
            DN[y,x] = np.nan if pixel is None else pixel
            Modified_DN[y,x] = np.nan if pixel_modified is None else pixel_modified
            Mask[y,x] = np.nan if mask is None else mask
    return DN, Modified_DN, Mask

In [74]:
base_dir = "/Volumes/Transcend/tileDB"
os.makedirs(base_dir, exist_ok=True)
array_uri = os.path.join(base_dir, "Haya2TIRTest")

In [81]:
domain = tiledb.Domain(
    tiledb.Dim(name="file_idx", domain=(0, 100000), dtype=np.int64),  # ファイルの整数インデックス
    tiledb.Dim(name="x", domain=(0, WIDTH - 1), dtype=np.int32),
    tiledb.Dim(name="y", domain=(0, HEIGHT - 1), dtype=np.int32),
)
filename =tiledb.Attr(name="filename", dtype=str)
pixel = tiledb.Attr(name="pixel", dtype=np.int32)
pixel_modified = tiledb.Attr(name="pixel_modified", dtype=np.int32)
mask = tiledb.Attr(name="mask", dtype=np.int32)
attrs = (filename, pixel, pixel_modified, mask) 

In [76]:
schema = tiledb.ArraySchema(
    domain=domain,
    attrs=attrs,
    sparse=True
)


In [77]:
array_uri = os.path.join(base_dir, "Haya2TIRTest")
if tiledb.array_exists(array_uri):
    tiledb.remove(array_uri)
tiledb.Array.create(array_uri, schema)


In [80]:
# インデックスとファイル名の対応表をCSVに保存
file_index_mapping = pd.DataFrame({"file_idx": range(len(filenames)), "filename": filenames})
mapping_path = os.path.join(base_dir, "file_index_mapping.csv")
file_index_mapping.to_csv(mapping_path, index=False)
print(f"Saved: {mapping_path}")

Saved: /Volumes/Transcend/tileDB/file_index_mapping.csv


In [82]:
for file_idx, filename in enumerate(filenames):
    DN, Modified_DN, Mask = fetch_img_file(filename)
    yy, xx = np.meshgrid(np.arange(HEIGHT), np.arange(WIDTH), indexing="ij")
    with tiledb.open(array_uri, "w") as A:
        A[np.full(HEIGHT * WIDTH, file_idx, dtype=np.int64), xx.flatten(), yy.flatten()] = {
            "filename": np.full(HEIGHT * WIDTH, filename, dtype=object),
            "pixel": DN.astype(np.int32).flatten(),
            "pixel_modified": Modified_DN.astype(np.int32).flatten(),
            "mask": Mask.astype(np.int32).flatten(),
        }

